# case01: 仮名加工情報 事例再現デモ

⏱ **所要時間 約10分** ／ 🎯 **ゴール**: 個人データを「仮名加工情報」に加工し、加工後も目的の分析が成立することを体験する。

**この Notebook の流れ**
1. データを取得して中身を見る
2. 加工設計にもとづき加工する（会員ID→整理番号 ／ 生年月日→年代 ／ 住所→市区町村 ／ 直接識別子は削除）
3. 加工前後を比べる
4. 加工後データで分析する
5. 加工が仕様どおりか確認テスト

**対象**: 個人情報保護委員会事務局レポート事例編「1.1 事例1」— 食品オンライン通販の事業者A。設計の詳しい理由（なぜその加工にするか）は公開サイトの
[① 検討](https://gghatano.github.io/pets-seminar-01/) を正本とします。

> 教材用の合成データを使用します。実在の個人・値とは無関係で、法令適合性を保証するものではありません。> 上のメニュー「ランタイム > すべてのセルを実行」でも一気に実行できます。

## 1. 事例概要〔報告〕

- **事業者A**: 食品のオンライン通信販売事業者。新規に首都圏で実店舗事業（有機食品）を計画。
- **保有データ**: 顧客の属性＋購買履歴＋Webアクセス履歴。
- **新たな利用目的**: 地域ごとにどの顧客層（年代・性別）がどの商品に関心を持つかを分析し、**出店計画**を検討する。
- 従来の利用目的に含まれないため、**仮名加工情報**に加工して利用目的を変更する。

## 2. データの取得
GitHub 上の生成済みダミーデータ（3テーブル）を取得します。

In [ ]:
import re
import pandas as pd

DATA_BASE = (
    "https://raw.githubusercontent.com/gghatano/pets-seminar-01/main/data/case01_pseudonymized/raw"
)

customers = pd.read_csv(f"{DATA_BASE}/customers.csv")
purchases = pd.read_csv(f"{DATA_BASE}/purchases.csv")
web_access = pd.read_csv(f"{DATA_BASE}/web_access.csv")

print("customers :", customers.shape)
print("purchases :", purchases.shape)
print("web_access:", web_access.shape)
customers.head()

## 3. 加工前データの確認

3テーブルは会員IDで関連づきます（顧客 1 ──< 購買 / アクセス）。詳細な定義は [加工前テーブル定義書](https://gghatano.github.io/pets-seminar-01/case01_pseudonymized/03_table_definition_before/) を参照。

In [ ]:
print("[customers] 列:", list(customers.columns))
print("[purchases] 列:", list(purchases.columns))
print("[web_access] 列:", list(web_access.columns))
display(purchases.head(3))
display(web_access.head(3))

## 4. 列ごとの情報特性（要約）

詳細は [情報特性・機微度分類](https://gghatano.github.io/pets-seminar-01/case01_pseudonymized/04_column_classification/) が正本。

| 項目 | 情報分類 | 識別性 | 分析上の必要性 |
|------|----------|--------|----------------|
| 会員ID | 内部管理識別子 | 低（単体）| 個人単位の履歴結合に必要 |
| 氏名 / 携帯 / メール / カード番号 | 直接識別子・到達性 | 高 | 不要 |
| 生年月日・住所 | 準識別子 | 組合せで中〜高 | 年代・地域として必要 |
| 性別 | 準識別子 | 低 | 必要 |
| Cookie ID | 内部管理識別子 | 低（到達性あり）| 不要 |
| 購入・アクセス履歴 | 行動・履歴情報 | 低 | 必要（分析対象）|

## 5. 利用目的と必要粒度〔報告〕
- 居住地域 → **市区町村単位**（商圏判定）
- 年齢 → **10歳区切りの年代**
- 購入・アクセス履歴 → できる限り加工しない

## 6. 加工方針（要約、図表1-3）〔報告〕
会員ID→整理番号（置換）／氏名・郵便番号・携帯・メール・カード番号・Cookie ID→削除／生年月日→年代・住所→市区町村（一般化）／性別・購入/アクセス履歴→加工なし。

## 7. 加工処理

[加工仕様書](https://gghatano.github.io/pets-seminar-01/case01_pseudonymized/06_processing_spec/) に沿って段階的に実行します。

### 7-1. 整理番号の付与（会員ID の置換）
会員IDを一意な整理番号に対応づけ、3テーブルで同じ整理番号を用いて個人単位の履歴結合を維持します。元の会員IDは加工後データから除去します。

In [ ]:
id_map = {mid: f"R{i + 1:06d}" for i, mid in enumerate(sorted(customers["会員ID"]))}
print("対応の例:", list(id_map.items())[:3])

### 7-2. 生年月日 → 年代（10歳区切りに一般化）
年齢は固定の基準年で算出します（教材の再現性のため）。

In [ ]:
REFERENCE_YEAR = 2026  # 生成データの基準（src/dummy_data の REFERENCE_DATE と一致）


def to_decade(birth: str) -> str:
    age = REFERENCE_YEAR - int(str(birth)[:4])
    return f"{min(age // 10 * 10, 80)}代"  # 80歳以上は「80代」


print(to_decade("1985-04-12"), to_decade("2009-01-01"), to_decade("1940-12-31"))

### 7-3. 住所 → 市区町村（丁目以降を削除して一般化）
住所は丁目以降（数字始まり）を除去し、政令市は「◯◯市◯◯区」までを残します。

In [ ]:
def to_city(addr: str) -> str:
    return re.sub(r"\d.*$", "", addr)  # 最初の数字（丁目）以降を除去


print(to_city("東京都世田谷区3丁目12-5"))
print(to_city("神奈川県横浜市青葉区2丁目8-1"))

### 7-4. 加工後テーブルの組み立て（不要・直接識別列の削除）

In [ ]:
customers_p = pd.DataFrame(
    {
        "整理番号": customers["会員ID"].map(id_map),
        "性別": customers["性別"],
        "年代": customers["生年月日"].map(to_decade),
        "市区町村": customers["住所"].map(to_city),
    }
)
purchases_p = purchases.assign(整理番号=purchases["会員ID"].map(id_map))[
    ["購買ID", "整理番号", "購入年月日", "購入品目", "購入数量", "購入金額"]
]
web_access_p = web_access.assign(整理番号=web_access["会員ID"].map(id_map))[
    ["アクセスID", "整理番号", "アクセス日時", "閲覧カテゴリ"]
]
customers_p.head()

## 8. 加工前後の比較
何が削除・一般化・維持されたかを確認します。

In [ ]:
print("加工前 customers 列:", list(customers.columns))
print("加工後 customers 列:", list(customers_p.columns))
removed = set(customers.columns) - set(customers_p.columns) | {"Cookie_ID"}
print("削除された属性/識別子:", sorted(removed))
print("生年月日 -> 年代 / 住所 -> 市区町村 に一般化, 会員ID -> 整理番号 に置換")
display(pd.concat([customers.head(2), customers_p.head(2)], axis=1))

## 9. 加工後データによる分析
加工後データでも、利用目的（地域×顧客層×商品関心）の分析が成立することを確認します。

In [ ]:
m = purchases_p.merge(customers_p, on="整理番号")

print("■ 年代別 平均購入金額")
display(m.groupby("年代")["購入金額"].mean().round(0))

print("■ 市区町村別 平均購入金額（上位5）")
display(m.groupby("市区町村")["購入金額"].mean().round(0).sort_values(ascending=False).head(5))

print("■ 年代別 購入品目トップ（若年 vs 高年）")
for d in ["20代", "70代"]:
    top = m[m["年代"] == d]["購入品目"].value_counts(normalize=True).head(3)
    print(d, list(top.index))

### 9-1. ここを変えて実験してみよう 🧪

下の `対象カテゴリ` を別の商品（例: `"菓子"`, `"鮮魚"`, `"米・穀物"`）に変えて再実行すると、その商品を買った人の**年代構成**が変わります。加工後データ（整理番号・年代・市区町村だけ）でも、こうした顧客層の分析ができることを確かめましょう。

In [ ]:
対象カテゴリ = "有機食品セット"  # ← ここを "菓子" や "鮮魚" などに変えて再実行

購入者 = m[m["購入品目"] == 対象カテゴリ]
print(f"「{対象カテゴリ}」を買った人の年代構成（％）")
display((購入者["年代"].value_counts(normalize=True).sort_index() * 100).round(1))

## 10. 確認テスト
加工仕様どおりに処理されたことを assert で確認します。

In [ ]:
deleted = {
    "氏名",
    "郵便番号",
    "住所",
    "携帯電話番号",
    "電子メールアドレス",
    "クレジットカード番号",
    "会員ID",
}
assert deleted.isdisjoint(customers_p.columns), "削除対象列が残っている"
assert "Cookie_ID" not in web_access_p.columns and "会員ID" not in web_access_p.columns
assert customers_p["整理番号"].is_unique and len(customers_p) == len(customers)
assert len(purchases_p) == len(purchases) and len(web_access_p) == len(web_access)
assert set(purchases_p["整理番号"]).issubset(set(customers_p["整理番号"]))
assert set(customers_p["年代"]).issubset({f"{d}代" for d in range(10, 90, 10)})
print("すべての確認テストに合格しました。")

## 11. まとめ

データ理解 → 情報特性の評価 → 利用目的の整理 → 加工設計 → **加工の実行** → 加工前後の比較 → 加工後データの利用、という一連の流れを再現しました。

- 直接識別子・本人到達性のある項目（氏名・携帯・メール・カード番号・Cookie ID）は削除。
- 生年月日・住所は、分析に必要な粒度（年代・市区町村）まで一般化。
- 会員IDは整理番号に置換し、個人単位の履歴結合を維持。
- 加工後も、地域×年代×性別 の購買傾向分析が成立。

詳細な設計・根拠は [公開サイト](https://gghatano.github.io/pets-seminar-01/) を参照してください。

**次に**: [③ 結果ページ](https://gghatano.github.io/pets-seminar-01/case01_pseudonymized/09_results/) で、加工前後の比較・確認テスト・分析結果をまとめて確認できます。